# 00 — Data Validation

Confirms that all data sources load correctly end-to-end before any analysis begins.

**Run this notebook before starting any analysis work.** It validates:
1. FastF1 cache initialises and a session loads
2. OpenF1 API is reachable and returns expected data
3. Era helper correctly maps years to era names and years-within-era
4. Core data models instantiate without errors

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

## 1. Era Helper

In [ ]:
from src.utils.era_helper import get_era_info

test_years = [2022, 2023, 2024, 2025, 2026]
for year in test_years:
    info = get_era_info(year)
    print(f"{year} → {info.name}, Year {info.year_within_era} of era")

Expected output:
```
2022 → Ground Effect Era, Year 1 of era
2023 → Ground Effect Era, Year 2 of era
2024 → Ground Effect Era, Year 3 of era
2025 → Ground Effect Era, Year 4 of era
2026 → 2026 Era, Year 1 of era
```

## 2. FastF1 — Load a Race Session

Loads the 2022 Bahrain Grand Prix (Race). First run will populate the cache — expect 1–2 minutes. Subsequent runs are near-instant.

In [ ]:
from src.data.fastf1_loader import load_session, get_race_laps

session = load_session(2022, 'Bahrain', 'R')
print(f"Session loaded: {session.event['EventName']} {session.event.year}")
print(f"Session type:   {session.name}")
print(f"Date:           {session.date}")

In [ ]:
laps = get_race_laps(2022, 'Bahrain')
print(f"Total laps in session: {len(laps)}")
print(f"Drivers:               {laps['Driver'].nunique()}")
print(f"\nSample lap data:")
laps[['Driver', 'LapNumber', 'LapTime', 'Compound', 'TyreLife']].head(10)

In [ ]:
# Fastest lap per driver
fastest = laps.groupby('Driver')['LapTime'].min().sort_values().reset_index()
fastest.columns = ['Driver', 'Fastest Lap']
fastest['Fastest Lap (s)'] = fastest['Fastest Lap'].dt.total_seconds()
print("Fastest lap per driver (2022 Bahrain GP):")
fastest[['Driver', 'Fastest Lap (s)']].head(10)

## 3. FastF1 — Event Schedule

In [ ]:
from src.data.fastf1_loader import get_event_schedule

schedule = get_event_schedule(2022)
print(f"2022 season: {len(schedule)} events")
schedule[['RoundNumber', 'EventName', 'EventDate', 'Country']].head(10)

## 4. OpenF1 — Sessions and Drivers

Confirms the API is reachable and returns structured data.

In [ ]:
from src.data.openf1_client import get_sessions, get_drivers

sessions_2024 = get_sessions(2024, session_type='Race')
print(f"2024 Race sessions returned: {len(sessions_2024)}")
if sessions_2024:
    print(f"\nFirst session:")
    first = sessions_2024[0]
    print(f"  Name:        {first.get('session_name')}")
    print(f"  Date:        {first.get('date_start')}")
    print(f"  Session key: {first.get('session_key')}")

In [ ]:
if sessions_2024:
    session_key = sessions_2024[0]['session_key']
    drivers = get_drivers(session_key)
    print(f"Drivers in session {session_key}: {len(drivers)}")
    for d in drivers[:5]:
        print(f"  #{d.get('driver_number')} {d.get('full_name')} ({d.get('team_name')})")

## 5. Data Models — Smoke Test

Instantiates each model with representative data to confirm they work as expected.

In [ ]:
from src.models import RaceResult, LapSummary, DriverStanding, ConstructorStanding

result = RaceResult(
    season=2022, round=1, race_name='Bahrain Grand Prix',
    driver_id='VER', driver_name='Max Verstappen',
    constructor_id='red_bull', constructor_name='Red Bull Racing',
    grid_position=1, finish_position=None, points=0.0,
    status='DNF', fastest_lap=False, laps_completed=40,
)
print(f"RaceResult — classified: {result.classified_finish}, DNF: {result.dnf}")

lap = LapSummary(
    season=2022, round=1, session_type='R',
    driver_id='LEC', driver_name='Charles Leclerc',
    constructor_id='ferrari',
    fastest_lap_time_s=94.112, median_lap_time_s=95.8,
    total_laps=57, pit_stops=2, tyre_compounds_used=['SOFT', 'MEDIUM'],
)
print(f"LapSummary  — valid lap: {lap.has_valid_lap}, fastest: {lap.fastest_lap_time_s}s")

print("\nAll models instantiated successfully.")

## Validation Summary

If all cells above ran without errors:

- ✅ Era helper: year → era name + year-within-era
- ✅ FastF1: cache enabled, session loads, laps accessible
- ✅ OpenF1: API reachable, sessions and drivers returned
- ✅ Data models: all four instantiate correctly

**The foundation is ready. Proceed to `01_2022_era_baseline.ipynb`.**